# Predicting the Ballon d'Or during the season 2024-2025 : 

In [6]:
import pandas as pd
import numpy as np

print("Chargement  des données des joueurs...")
df_players = pd.read_csv('players_data-2024_2025.csv')

def map_position(pos_str):
    if pos_str in ['GK']:
        return 'Goalkeeper'
    elif pos_str in ['CB', 'LB', 'RB', 'LWB', 'RWB']:
        return 'Defender'
    elif pos_str in ['CDM', 'CM', 'CAM']:
        return 'Midfielder'
    elif pos_str in ['LW', 'RW', 'CF', 'ST']:
        return 'Forward'
    else:
        return 'Unknown'

df_players['Position_Category'] = df_players['Pos'].apply(map_position)

df_filtered = df_players[df_players['Min'] >= 1500].copy()

df_filtered['Gls_per_90'] = df_filtered['Gls'] / (df_filtered['Min'] / 90)
df_filtered['Ast_per_90'] = df_filtered['Ast'] / (df_filtered['Min'] / 90)
df_filtered['xG_per_90'] = df_filtered['xG'] / (df_filtered['Min'] / 90)

position_weights = {
    'Goalkeeper': {'Gls_per_90': 0.1, 'Ast_per_90': 0.1, 'xG_per_90': 0.1},
    'Defender': {'Gls_per_90': 0.2, 'Ast_per_90': 0.3, 'xG_per_90': 0.2},
    'Midfielder': {'Gls_per_90': 0.3, 'Ast_per_90': 0.4, 'xG_per_90': 0.3},
    'Forward': {'Gls_per_90': 0.5, 'Ast_per_90': 0.3, 'xG_per_90': 0.5}
}


for metric in ['Gls_per_90', 'Ast_per_90', 'xG_per_90']:
    min_val = df_filtered.groupby('Position_Category')[metric].transform('min')
    max_val = df_filtered.groupby('Position_Category')[metric].transform('max')
    df_filtered[f'norm_{metric}'] = (df_filtered[metric] - min_val) / (max_val - min_val)


def compute_row_boi(row):
    w = position_weights.get(row['Position_Category'], position_weights['Forward'])
    return (row['norm_Gls_per_90'] * w['Gls_per_90'] +
            row['norm_Ast_per_90'] * w['Ast_per_90'] +
            row['norm_xG_per_90'] * w['xG_per_90']
    )

df_filtered['BOI'] = df_filtered.apply(compute_row_boi, axis=1)


top_clubs = ['Real Madrid', 'Barcelona', 'Manchester City', 'Bayern Munich', 'Liverpool', 'Paris S-G']
df_filtered['club_boost'] = df_filtered['club'].apply(lambda x: 1.2 if any(c in x for c in top_clubs) else 1)


df_filtered['final_score'] = df_filtered['BOI'] * df_filtered['club_boost']
df_filtered['Predicted_Rank'] = df_filtered['final_score'].rank(ascending=False, method='min').astype(int)

results = df_filtered.sort_values(by='Predicted_Rank') 

print("Top 10 Players by Predicted Rank:")
print(results[['Predicted_Rank', 'Player', 'Squad', 'Comp', 'Pos', 'Gls', 'Ast', 'Final_Score']].head(10).to_string(index=False))




Chargement  des données des joueurs...


FileNotFoundError: [Errno 2] No such file or directory: 'players_data-2024_2025.csv'